# opyf_colab — Hydraulic Analysis Pipeline

**Brague flood event reconstruction** using monocular depth, JAX LSPIV surface-velocity
measurement, D8 thalweg extraction, and IS 10430 canal design optimisation.

> Generated by `make_notebook.py`. Execute top-to-bottom with Papermill for a
> fully reproducible run, or run individual cells interactively.

---


## Parameters
All paths and flags used throughout the notebook. Tagged `parameters` for Papermill override.

In [ ]:
# -- paths --
VIDEO_DOWN   = "data/brague/IMG_1139.MOV"   # downstream bridge
VIDEO_UP     = "data/brague/IMG_1142.MOV"   # upstream bridge
MNT_XYZ      = "data/brague/MNT.xyz"
ORTHO_TIF    = "data/brague/Ortho.tif"
DP_WEIGHTS   = "weights/depth_pro.msgpack"
SP_WEIGHTS   = "weights/superpoint.msgpack"
LG_WEIGHTS   = "weights/superpoint_lightglue.msgpack"
OUT_DIR      = "output/brague"
ASSETS_DIR   = "assets"
N_FRAMES     = 5

# -- skip flags (set True to re-use cached outputs) --
SKIP_DOWNLOAD  = True
SKIP_DEPTH     = True
SKIP_ALIGN     = True
SKIP_D8        = False
SKIP_PC_CHECK  = False
SKIP_CANAL     = False
SKIP_CANAL_VIZ = False
SKIP_LSPIV     = False
SKIP_VIZ       = False


## Environment setup

In [ ]:
import os, sys, json, time
from pathlib import Path

# JAX on CPU (set before importing jax)
os.environ.setdefault("JAX_PLATFORMS", "cpu")

REPO = Path(".").resolve()
os.chdir(REPO)
print("Working dir:", REPO)
print("Python:", sys.executable)


## Stage 0 — Download release assets

Downloads all required large files (videos, weights, MNT point cloud, orthorectified
GeoTIFF) from the GitHub release if not already present locally.


In [ ]:
if not SKIP_DOWNLOAD:
    from pipeline import download_assets
    download_assets()
else:
    print("Stage 0 skipped — assets assumed present")


## Stages 1–5 — Depth inference pipeline

1. **Extract frames** from the event video at uniform intervals
2. **Load Depth Pro** weights (Apple's monocular metric depth model)
3. **Estimate dry mask** from the pre-event orthorectified GeoTIFF
4. **Rasterise MNT** bed elevation onto the ortho grid
5. **Per-frame inference**: Depth Pro → inverse depth → metric scale alignment via GCPs → flow depth _h_

Outputs `output/brague/flow_depth.tif` + per-frame `*_z_surface.tif`.


In [ ]:
meta_path = Path(OUT_DIR) / "pipeline_meta.json"
flow_tif  = Path(OUT_DIR) / "flow_depth.tif"

if SKIP_DEPTH and flow_tif.exists() and meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    print(f"Depth pipeline skipped — loaded {meta_path}")
    print(f"  h_mean={meta['h_final_mean']:.3f} m  h_max={meta['h_final_max']:.3f} m")
else:
    import types
    args = types.SimpleNamespace(
        video=VIDEO_DOWN, mnt=MNT_XYZ, ortho=ORTHO_TIF,
        weights=DP_WEIGHTS, out_dir=OUT_DIR,
        n_frames=N_FRAMES, sp_weights=SP_WEIGHTS, lg_weights=LG_WEIGHTS,
    )
    from pipeline import run_depth_pipeline
    meta = run_depth_pipeline(args)


## Stage 6b — Point cloud alignment + water volume

Aligns the Depth Pro point cloud from the first video frame to the Lambert-93
MNT using LightGlue feature matching. Estimates inundated volume and area.

Output: `assets/alignment_*.png`


In [ ]:
if not SKIP_ALIGN:
    import numpy as np, rasterio as _rio
    from modules.depth_to_elevation import load_mnt, rasterise_mnt
    from pipeline import run_alignment_stage

    with _rio.open(ORTHO_TIF) as src:
        _transform   = src.transform
        _ortho_shape = (src.height, src.width)
    X_mnt, Y_mnt, Z_mnt = load_mnt(MNT_XYZ, subsample=5)
    z_bed = rasterise_mnt(X_mnt, Y_mnt, Z_mnt, _transform, _ortho_shape)

    align_result = run_alignment_stage(
        Path(OUT_DIR), z_bed, _transform, X_mnt, Y_mnt, Z_mnt,
        Path(ASSETS_DIR), SP_WEIGHTS, LG_WEIGHTS,
    )
    if align_result:
        print(f"Water volume  : {align_result['volume_m3']:,.0f} m³")
        print(f"Inundated area: {align_result['area_m2']:,.0f} m²")
else:
    print("Stage 6b skipped")


## Stage 6c — D8 thalweg extraction

Extracts the deepest-flow path through the inundated zone using the D8 flow
direction algorithm (JAX-accelerated). Outputs:
- `centerline_x/y` Lambert-93 thalweg coordinates
- Longitudinal slope profile
- `assets/d8_thalweg.png`


In [ ]:
d8_result = None
if not SKIP_D8:
    from pipeline import run_d8_thalweg
    d8_result = run_d8_thalweg(Path(OUT_DIR), Path(ASSETS_DIR))
    if d8_result:
        geo = d8_result["geometry"]
        print(f"Thalweg length : {geo['length_m']:.1f} m")
        print(f"Mean bed slope : {geo['slope_mean']:.5f}")
else:
    print("Stage 6c skipped")


## Stage 6d — Point cloud × ortho alignment check

Drapes the Ortho.tif RGB colours onto MNT.xyz elevation points and produces a
4-panel diagnostic figure showing the spatial alignment quality:

- Panel A: 3D coloured point cloud (Ortho RGB on MNT Z)
- Panel B: Top-down ortho + MNT elevation contours
- Panel C: Plan-view elevation heatmap
- Panel D: Elevation histogram split by vegetation / water / urban colour class

Output: `assets/pointcloud_ortho_check.png`


In [ ]:
if not SKIP_PC_CHECK:
    from pipeline import run_pc_ortho_check
    run_pc_ortho_check(MNT_XYZ, ORTHO_TIF, Path(ASSETS_DIR))
else:
    print("Stage 6d skipped")


In [ ]:
from IPython.display import Image, display
pc_check_img = Path(ASSETS_DIR) / "pointcloud_ortho_check.png"
if pc_check_img.exists():
    display(Image(str(pc_check_img), width=900))


## Stage 7 — JAX canal optimiser (IS 10430)

Optimises a trapezoidal canal cross-section for the flood discharge using the
Manning-Strickler equation and IS 10430:2000 velocity limits. Finds the minimum
wetted-perimeter section satisfying:
- Manning's n for concrete lining
- IS velocity bounds (0.6–3.0 m/s for concrete)
- IS freeboard table

Output: `canal_design/canal_params.json`


In [ ]:
canal_dir = Path("canal_design")
cp_path   = canal_dir / "canal_params.json"

if SKIP_CANAL and cp_path.exists():
    with open(cp_path) as f:
        canal_params = json.load(f)
    print("Canal optimizer skipped — loaded", cp_path)
else:
    from pipeline import run_canal_optimizer
    canal_params = run_canal_optimizer(meta, canal_dir, Path(ASSETS_DIR))

print(f"  B = {canal_params['bed_width_m']:.3f} m")
print(f"  D = {canal_params['water_depth_m']:.3f} m")
print(f"  Q = {canal_params['Q_calculated_m3s']:.2f} m³/s")
print(f"  V = {canal_params['velocity_ms']:.3f} m/s")
print(f"  n = {canal_params['manning_n']:.4f}")
print(f"  S = {canal_params['long_slope']:.6f}")


In [ ]:
# Display canal_section.png and design_chain.png
for fname in ("canal_section.png", "design_chain.png"):
    img_path = Path(ASSETS_DIR) / fname
    if img_path.exists():
        print(fname)
        display(Image(str(img_path), width=700))


## Stage 7b — Canal 3D overlay

Renders the designed canal section over the reconstructed terrain and ortho
background, with the D8 thalweg overlaid if available.

Output: `assets/canal_3d_overlay.png`


In [ ]:
reach_geo = None
if not SKIP_CANAL_VIZ:
    from pipeline import run_canal_3d_viz
    reach_geo = run_canal_3d_viz(
        canal_params, Path(OUT_DIR), ORTHO_TIF, Path(ASSETS_DIR), d8_result,
    )
else:
    print("Stage 7b skipped")


In [ ]:
overlay_img = Path(ASSETS_DIR) / "canal_3d_overlay.png"
if overlay_img.exists():
    display(Image(str(overlay_img), width=900))


## Stage 7c — Canal 4-view CAD drawing

Generates a 4-view engineering drawing (Front / Side / Top / Isometric) with
dimension lines and annotations for the optimised canal cross-section.

Output: `assets/canal_cad_model.png`


In [ ]:
if not SKIP_CANAL_VIZ:
    from pipeline import run_canal_cad
    run_canal_cad(canal_params, reach_geo, Path(ASSETS_DIR))
else:
    print("Stage 7c skipped")


In [ ]:
cad_img = Path(ASSETS_DIR) / "canal_cad_model.png"
if cad_img.exists():
    display(Image(str(cad_img), width=900))


## Stage 7d — JAX LSPIV surface-velocity + discharge

Runs the full LSPIV (Large-Scale Particle Image Velocimetry) pipeline on
consecutive video frames from **both** bridge camera positions:

- **IMG_1139** (downstream bridge): GCPs from known bridge geometry
- **IMG_1142** (upstream bridge): GCPs from upstream bridge geometry

Pipeline per video:
1. Orthorectification via DLT homography (4+ GCP correspondences)
2. FFT phase-correlation PIV on 32×32 px interrogation windows
3. Gaussian sub-pixel peak localisation
4. Gaussian RBF interpolation onto uniform grid
5. MNT transect sampling → depth profile
6. Discharge integration Q = α ∫ V · h dl  (α = 0.9)

Also generates three **opyflow-equivalent figures** that replicate the code blocks
from the [original opyflow notebook](https://github.com/groussea/opyflow/blob/master/tests/Test_Brague_flood/test_opyf_LSPIV_Brague.md)
using our JAX pipeline:

| opyflow original | JAX equivalent |
|---|---|
| `birdEyeTransf1139.png` | `assets/opyflow_birdeye.png` |
| `1139.png` + `1142.png` | `assets/opyflow_velocity_field.png` |
| `figure_Brague.png` | `assets/figure_brague.png` |

Outputs: `assets/lspiv_results.png`, `assets/opyflow_birdeye.png`,
`assets/opyflow_velocity_field.png`, `assets/figure_brague.png`


In [ ]:
lspiv_result = None
if not SKIP_LSPIV:
    from pipeline import run_lspiv
    lspiv_result = run_lspiv(
        video_down=VIDEO_DOWN,
        video_up=VIDEO_UP,
        mnt_xyz=MNT_XYZ,
        ortho_tif=ORTHO_TIF,
        out_dir=Path(OUT_DIR),
        assets=Path(ASSETS_DIR),
    )
    if lspiv_result:
        print(f"Combined discharge Q = {lspiv_result['Q']:.2f} m³/s")
else:
    print("Stage 7d skipped")


In [ ]:
# 6-panel LSPIV summary
lspiv_img = Path(ASSETS_DIR) / "lspiv_results.png"
if lspiv_img.exists():
    display(Image(str(lspiv_img), width=900))


### opyflow-equivalent figures (JAX reimplementation)

Replicates the three figure code blocks from the original opyflow Brague notebook
using JAX outputs and the same GCPs / input data.


In [ ]:
# birdEyeTransf1139.png equivalent
birdeye = Path(ASSETS_DIR) / "opyflow_birdeye.png"
if birdeye.exists():
    print("Bird-eye orthorectified frames (birdEyeTransf1139.png equivalent)")
    display(Image(str(birdeye), width=900))


In [ ]:
# 1139.png + 1142.png equivalent
vel_field = Path(ASSETS_DIR) / "opyflow_velocity_field.png"
if vel_field.exists():
    print("Velocity colour field (1139.png + 1142.png equivalent)")
    display(Image(str(vel_field), width=900))


In [ ]:
# figure_Brague.png equivalent
fig_brague = Path(ASSETS_DIR) / "figure_brague.png"
if fig_brague.exists():
    print("figure_Brague.png equivalent — scatter + transect + Q")
    display(Image(str(fig_brague), width=900))


## Stage 8 — Annotated pipeline visualisation

Builds a combined 8-panel summary figure covering the full pipeline from raw
video frame through depth estimation, flow depth, canal design, and LSPIV
discharge.

Output: `assets/annotated_pipeline.png`


In [ ]:
if not SKIP_VIZ:
    from pipeline import run_visualisation
    run_visualisation(Path(ASSETS_DIR))
else:
    print("Stage 8 skipped")


In [ ]:
for fname in ("annotated_pipeline.png", "future_roadmap.png", "multiview_comparison.png"):
    img_path = Path(ASSETS_DIR) / fname
    if img_path.exists():
        print(fname)
        display(Image(str(img_path), width=900))


## Results summary


In [ ]:
print("=" * 60)
print("  opyf_colab Pipeline — Key Results")
print("=" * 60)
print(f"  Flow depth h_mean     = {meta.get('h_final_mean', float('nan')):.3f} m")
print(f"  Flow depth h_max      = {meta.get('h_final_max', float('nan')):.3f} m")
if d8_result:
    geo = d8_result["geometry"]
    print(f"  Thalweg length        = {geo['length_m']:.1f} m")
    print(f"  Mean bed slope        = {geo['slope_mean']:.5f}")
print(f"  Canal bed width       = {canal_params['bed_width_m']:.3f} m")
print(f"  Canal water depth     = {canal_params['water_depth_m']:.3f} m")
print(f"  Canal Q (IS 10430)    = {canal_params['Q_calculated_m3s']:.2f} m³/s")
if lspiv_result:
    print(f"  LSPIV discharge Q     = {lspiv_result['Q']:.2f} m³/s")
print("=" * 60)
